In [1]:
import pandas as pd

file_loc = '/mnt/disks/filedisk2a/Qingcheng/'
train_df   = pd.read_csv(file_loc + 'train_df.csv')
holdout_df = pd.read_csv(file_loc + 'holdout_df.csv')
print('train_df  :', train_df.shape)
print('holdout_df:', holdout_df.shape)
train_df.head()

train_df  : (582105, 78)
holdout_df: (222843, 78)


,constraint_family_num,dt,long_profit,short_profit,long_profit_sum_3d,short_profit_sum_3d,long_profit_sum_7d,short_profit_sum_7d,long_profit_sum_1m,short_profit_sum_1m,...,rt_nz_cnt_3m,rt_nz_cnt_1y,rt_nz_cnt_3y,rt_spike_7d,rt_spike_1m,rt_spike_3m,rt_spike_1y,rt_spike_3y,KV,KV_group
0,7,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,3.0,0.000000,0.000000,0.000000,0.000000,-852.205775,161.0,04_345
1,11,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,28232.886036,-77974.874419,...,1.0,5.0,5.0,0.000000,-2829.423183,-2894.467394,-1825.074444,-1841.973281,161.0,04_345
2,27,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,30.0,0.000000,0.000000,0.000000,0.000000,-909.199441,NaN,03_138
3,30,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,13.0,0.000000,0.000000,0.000000,0.000000,-747.833083,138.0,03_138
4,41,2021-07-01,0.0,0.0,9333.357069,-1297.486966,11249.508162,-2176.316475,965094.429461,-593286.123159,...,38.0,65.0,67.0,-744.890648,-808.141129,-1218.259754,-2271.628633,-2520.698961,115.0,02_<=115


In [2]:
# merge daily physical variables (wind / load / genoutage / ice price forecast) onto the splits
import sys
sys.path.append("/var/www/python/Prod/nighthawk/")
import pandas as pd, numpy as np
from nighthawk.data.pipeline.common_functions import wind, load, genoutage
from nighthawk.data.pipeline.var_handler import ice_elec_price_vh

OPEX = 'SPP'

# date range covering both splits
dts = pd.concat([train_df['dt'], holdout_df['dt']]).astype(str)
start_dt, end_dt = dts.min(), dts.max()
print('phys date range:', start_dt, '->', end_dt)

def _daily(df, col, name):
    """hourly forecast -> daily mean, dt as 'YYYY-MM-DD' string"""
    df = df.copy()
    df['dt'] = pd.to_datetime(df['dt']).dt.strftime('%Y-%m-%d')
    return df.groupby('dt', as_index=False)[col].mean().rename(columns={col: name})

# --- wind / load / genoutage daily forecast ---
wind_df  = wind.Wind(OPEX).get_total_wind(start_dt, end_dt, var_spec=['f'], impute=True)
wind_daily = _daily(wind_df, 'spp_wind_total_forecast_f', 'wind_forecast')

load_df  = load.Load(OPEX).get_total_load(start_dt, end_dt, var_spec=['f'], impute=True)
load_daily = _daily(load_df, 'spp_load_total_forecast_f', 'load_forecast')

go_df    = genoutage.GenOutage(OPEX).get_genoutage_by_level(start_dt, end_dt, var_spec=['f'], area_list=['SPP'])
go_col   = [c for c in go_df.columns if c.endswith('_forecast_f')][0]
genoutage_daily = _daily(go_df, go_col, 'genoutage_forecast')

# --- ice price forecast (INDIANAHUB proxy for SPP, node 636) ---
ice_df, _ = ice_elec_price_vh.get_data_and_mapping_for_ice_elec(
    [636], OPEX, ['INDIANAHUB'], start_dt, end_dt, var_spec=['f'], impute=True)
ice_daily = _daily(ice_df, 'INDIANAHUB_ice_elec_price_forecast_f', 'ice_price_forecast')

# --- combine all daily physical variables ---
phys_daily = (wind_daily
              .merge(load_daily,      on='dt', how='outer')
              .merge(genoutage_daily, on='dt', how='outer')
              .merge(ice_daily,       on='dt', how='outer'))
print('phys_daily:', phys_daily.shape)
display(phys_daily.head())

# --- merge onto train / holdout (dt-level) ---
train_df['dt']   = train_df['dt'].astype(str)
holdout_df['dt'] = holdout_df['dt'].astype(str)
train_df   = train_df.merge(phys_daily, on='dt', how='left')
holdout_df = holdout_df.merge(phys_daily, on='dt', how='left')
print('train_df  :', train_df.shape, '| holdout_df:', holdout_df.shape)
train_df.head()

phys date range: 2021-06-02 -> 2026-05-19
phys_daily: (1813, 5)


,dt,wind_forecast,load_forecast,genoutage_forecast,ice_price_forecast
0,2021-06-02,1916.375000,26712.708333,11762.216667,26.790000
1,2021-06-03,3768.707917,28603.416667,11286.708333,28.130000
2,2021-06-04,9523.051667,30291.458333,11359.608333,29.570000
3,2021-06-05,13756.800417,29792.250000,11398.025000,29.783333
4,2021-06-06,13693.260000,29196.708333,11431.691667,34.360000


train_df  : (582105, 82) | holdout_df: (222843, 82)


,constraint_family_num,dt,long_profit,short_profit,long_profit_sum_3d,short_profit_sum_3d,long_profit_sum_7d,short_profit_sum_7d,long_profit_sum_1m,short_profit_sum_1m,...,rt_spike_1m,rt_spike_3m,rt_spike_1y,rt_spike_3y,KV,KV_group,wind_forecast,load_forecast,genoutage_forecast,ice_price_forecast
0,7,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-852.205775,161.0,04_345,3450.68,34594.75,8961.054167,35.99
1,11,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,28232.886036,-77974.874419,...,-2829.423183,-2894.467394,-1825.074444,-1841.973281,161.0,04_345,3450.68,34594.75,8961.054167,35.99
2,27,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-909.199441,NaN,03_138,3450.68,34594.75,8961.054167,35.99
3,30,2021-07-01,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-747.833083,138.0,03_138,3450.68,34594.75,8961.054167,35.99
4,41,2021-07-01,0.0,0.0,9333.357069,-1297.486966,11249.508162,-2176.316475,965094.429461,-593286.123159,...,-808.141129,-1218.259754,-2271.628633,-2520.698961,115.0,02_<=115,3450.68,34594.75,8961.054167,35.99


## Driver analysis — what moves long & short profit (family-day level)

Goal: rank the variables that drive **short_profit** (loss side) and **long_profit**, both as a
*level* (regression) and as *tail-loss events* (`short_profit < THRESH`), then translate the top
stable drivers into cut rules. Importance is judged on the **holdout** split (out-of-sample) and
cross-checked by SHAP **and** permutation importance — only drivers that agree are trusted.

**Leakage guard:** same-day realized `dam`/`rtm` are excluded (they are contemporaneous with the
loss). Only lagged/rolling history and pre-bid forecasts (FlowRatio, wind/load/genoutage/ice) are used.

In [6]:
# === 1. feature setup (family-day level) ===
KEYS    = ['dt', 'constraint_family_num']
TARGETS = ['short_profit', 'long_profit']
# same-day realized / id columns -> EXCLUDE (leakage or non-features)
LEAK = ['dam', 'rtm', 'da_mvalue', 'rt_mvalue', 'bad_short', 'rt_da', 'rt_binds',
        'oops_constraint_num', 'monitored_clean', 'contingency_clean',
        'monitored_name', 'node_name']

num_cols = train_df.select_dtypes('number').columns.tolist()
features = [c for c in num_cols if c not in KEYS + TARGETS + LEAK]
CATS     = [c for c in ['KV_group'] if c in train_df.columns]   # categorical driver(s)

# tag features so you can separate "it was bad recently" from real conditions
persistence = [c for c in features if 'profit_sum' in c]
condition   = [c for c in features if c not in persistence]

print(f"{len(features)} numeric features (+{len(CATS)} categorical: {CATS})")
print(f"\nPERSISTENCE ({len(persistence)}): {persistence}")
print(f"\nCONDITION   ({len(condition)}): {condition}")

# tail-loss flag (same threshold used in the source notebook)
THRESH = -50000
for d in (train_df, holdout_df):
    d['bad_short'] = (d['short_profit'] < THRESH).astype(int)
print(f"\nbad_short rate  train={train_df['bad_short'].mean():.4f}  holdout={holdout_df['bad_short'].mean():.4f}")

72 numeric features (+1 categorical: ['KV_group'])

PERSISTENCE (10): ['long_profit_sum_3d', 'short_profit_sum_3d', 'long_profit_sum_7d', 'short_profit_sum_7d', 'long_profit_sum_1m', 'short_profit_sum_1m', 'long_profit_sum_3m', 'short_profit_sum_3m', 'long_profit_sum_1y', 'short_profit_sum_1y']

CONDITION   (62): ['FlowRatio', 'ShadowPrice_min', 'ShadowPrice_sum', 'MinFlowLimit', 'MaxFlowLimit', 'dam_lag1', 'rtm_lag1', 'dam_sum_7d', 'rtm_sum_7d', 'dam_sum_1m', 'rtm_sum_1m', 'dam_sum_3m', 'rtm_sum_3m', 'dam_sum_1y', 'rtm_sum_1y', 'dam_sum_3y', 'rtm_sum_3y', 'rtda_1d', 'rtda_sum_7d', 'rtda_avg_7d', 'rtda_max_7d', 'rtda_min_7d', 'rtda_sum_1m', 'rtda_avg_1m', 'rtda_max_1m', 'rtda_min_1m', 'rtda_sum_3m', 'rtda_avg_3m', 'rtda_max_3m', 'rtda_min_3m', 'rtda_sum_1y', 'rtda_avg_1y', 'rtda_max_1y', 'rtda_min_1y', 'rtda_sum_3y', 'rtda_avg_3y', 'rtda_max_3y', 'rtda_min_3y', 'rtda_std_1m', 'rtda_std_3m', 'rtda_std_1y', 'rtda_std_3y', 'rt_max_7d', 'rt_max_1m', 'rt_max_3m', 'rt_max_1y', 'rt_max_3y', '

In [7]:
# === 2. model + importance helper (SHAP + permutation, judged on holdout) ===
import numpy as np, pandas as pd, lightgbm as lgb, shap
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, roc_auc_score

SHAP_N = 50_000   # sample holdout for SHAP/permutation speed

def _xy(df, target):
    X = df[features + CATS].copy()
    for c in CATS: X[c] = X[c].astype('category')
    y = df[target]
    m = y.notna()
    return X[m], y[m]

def driver_analysis(target, task='reg'):
    Xtr, ytr = _xy(train_df, target)
    Xho, yho = _xy(holdout_df, target)
    if task == 'reg':
        model = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=31,
                                  subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
                                  random_state=0, n_jobs=-1, verbose=-1)
        model.fit(Xtr, ytr, categorical_feature=CATS)
        print(f"[{target}] R2  train={r2_score(ytr, model.predict(Xtr)):.3f}  "
              f"holdout={r2_score(yho, model.predict(Xho)):.3f}")
        scoring = 'r2'
    else:
        model = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=31,
                                   subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
                                   random_state=0, n_jobs=-1, verbose=-1, class_weight='balanced')
        model.fit(Xtr, ytr, categorical_feature=CATS)
        print(f"[{target}] AUC train={roc_auc_score(ytr, model.predict_proba(Xtr)[:,1]):.3f}  "
              f"holdout={roc_auc_score(yho, model.predict_proba(Xho)[:,1]):.3f}")
        scoring = 'roc_auc'

    Xs = Xho.sample(min(SHAP_N, len(Xho)), random_state=0)
    ys = yho.loc[Xs.index]

    sv = shap.TreeExplainer(model).shap_values(Xs)
    if isinstance(sv, list): sv = sv[-1]
    elif getattr(sv, 'ndim', 2) == 3: sv = sv[:, :, -1]

    perm = permutation_importance(model, Xs, ys, scoring=scoring,
                                  n_repeats=3, random_state=0, n_jobs=-1)
    imp = pd.DataFrame({'shap': np.abs(sv).mean(0),
                        'perm': perm.importances_mean}, index=Xs.columns)
    imp['shap_rank'] = imp['shap'].rank(ascending=False)
    imp['perm_rank'] = imp['perm'].rank(ascending=False)
    imp['mean_rank'] = imp[['shap_rank', 'perm_rank']].mean(1)
    imp = imp.sort_values('mean_rank')
    return {'model': model, 'shap': sv, 'X': Xs, 'imp': imp}

short_res = driver_analysis('short_profit', 'reg')
long_res  = driver_analysis('long_profit',  'reg')
bad_res   = driver_analysis('bad_short',     'clf')

print('\n=== top drivers of SHORT profit (level) ==='); display(short_res['imp'].head(15).round(3))
print('=== top drivers of LONG profit (level) ===');   display(long_res['imp'].head(15).round(3))
print('=== top drivers of BAD-SHORT tail events ==='); display(bad_res['imp'].head(15).round(3))

[short_profit] R2  train=0.366  holdout=-0.021
[long_profit] R2  train=0.349  holdout=0.035
[bad_short] AUC train=0.982  holdout=0.938


/opt/venvs/prod-py312/lib/python3.12/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



=== top drivers of SHORT profit (level) ===


,shap,perm,shap_rank,perm_rank,mean_rank
long_profit_sum_1y,506.035,0.031,1.0,2.0,1.5
short_profit_sum_1m,471.961,0.012,3.0,4.0,3.5
short_profit_sum_3d,307.610,0.031,8.0,1.0,4.5
dam_sum_7d,480.835,0.003,2.0,13.0,7.5
rtda_sum_1y,229.161,0.006,14.0,6.0,10.0
long_profit_sum_3m,347.226,0.003,6.0,15.0,10.5
rtm_lag1,243.806,0.003,11.0,12.0,11.5
rtda_sum_3y,198.876,0.004,15.0,9.0,12.0
load_forecast,151.515,0.023,21.0,3.0,12.0
rt_nz_cnt_1m,176.871,0.005,19.0,7.0,13.0


=== top drivers of LONG profit (level) ===


,shap,perm,shap_rank,perm_rank,mean_rank
long_profit_sum_1y,666.771,0.009,1.0,1.0,1.0
long_profit_sum_3m,402.465,0.006,3.0,3.0,3.0
dam_lag1,340.963,0.004,6.0,4.0,5.0
rt_spike_1m,311.891,0.003,8.0,8.0,8.0
rtda_sum_3y,154.534,0.007,20.0,2.0,11.0
FlowRatio,329.858,0.001,7.0,16.0,11.5
dam_sum_7d,165.062,0.003,17.0,7.0,12.0
rtda_sum_7d,275.633,0.002,11.0,14.0,12.5
rtda_sum_3m,204.162,0.002,15.0,12.0,13.5
short_profit_sum_7d,152.908,0.004,22.0,6.0,14.0


=== top drivers of BAD-SHORT tail events ===


,shap,perm,shap_rank,perm_rank,mean_rank
FlowRatio,0.601,0.021,1.0,1.0,1.0
rtm_sum_1m,0.296,0.005,3.0,3.0,3.0
rtda_std_1m,0.388,0.005,2.0,4.0,3.0
wind_forecast,0.289,0.007,4.0,2.0,3.0
rtm_sum_7d,0.144,0.003,7.0,5.0,6.0
rtda_min_3m,0.136,0.001,8.0,12.0,10.0
rt_nz_cnt_1y,0.145,0.001,6.0,14.0,10.0
rtm_sum_1y,0.093,0.001,10.0,11.0,10.5
rtda_std_3y,0.081,0.001,12.0,10.0,11.0
rt_spike_3y,0.148,0.001,5.0,22.0,13.5


In [ ]:
# === 3. SHAP summary plots (direction + magnitude) ===
import matplotlib.pyplot as plt
for name, res in [('short_profit', short_res), ('long_profit', long_res), ('bad_short', bad_res)]:
    shap.summary_plot(res['shap'], res['X'], max_display=15, show=False)
    plt.title(f'SHAP — {name}'); plt.tight_layout(); plt.show()

In [ ]:
# === 4. control tradeoff: cutting the worst tail of a driver ===
# For a candidate driver, drop the family-days in its 'bad' tail and measure
# short loss avoided vs long profit sacrificed. Set high_is_bad from the SHAP plot.
def cut_tradeoff(df, feat, q=0.10, high_is_bad=True):
    s = df[feat]
    thr = s.quantile(1 - q) if high_is_bad else s.quantile(q)
    cut = (s >= thr) if high_is_bad else (s <= thr)
    return pd.Series({
        'feature': feat, 'threshold': round(thr, 3), 'cut_frac': round(cut.mean(), 3),
        'short_loss_avoided': -df.loc[cut, 'short_profit'].sum(),   # >0 = loss removed
        'long_profit_lost':    df.loc[cut, 'long_profit'].sum(),    # >0 = profit forgone
        'net_benefit':       -df.loc[cut, 'short_profit'].sum() - df.loc[cut, 'long_profit'].sum(),
    })

# evaluate the top tail-event drivers on the holdout (set high_is_bad per SHAP direction)
top_drivers = bad_res['imp'].head(6).index.tolist()
tradeoff = pd.DataFrame([cut_tradeoff(holdout_df, f, q=0.10, high_is_bad=True)
                         for f in top_drivers if f in holdout_df.columns])
display(tradeoff.sort_values('net_benefit', ascending=False))

In [8]:
# === where does the BAD-SHORT forest split? ===
import numpy as np, pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

bad_model = bad_res['model']
top_feats = bad_res['imp'].head(8).index.tolist()

# ---- A. exact split thresholds from the trained forest (gain-weighted) ----
tdf    = bad_model.booster_.trees_to_dataframe()
splits = tdf[tdf['split_feature'].notna()].copy()
splits['thr_num'] = pd.to_numeric(splits['threshold'], errors='coerce')  # NaN = categorical split

print("Split thresholds per top driver (gain-weighted, from the actual forest):\n")
rows = []
for f in top_feats:
    s = splits[splits['split_feature'] == f]
    if s.empty:
        rows.append({'feature': f, 'note': 'not used as a split'}); continue
    sn = s.dropna(subset=['thr_num'])
    if sn.empty:                                   # categorical (e.g. KV_group)
        rows.append({'feature': f, 'n_splits': len(s), 'note': 'categorical split'}); continue
    w = sn['split_gain']
    gain_wtd_thr = np.average(sn['thr_num'], weights=w)        # typical split point
    band = sn.groupby(pd.cut(sn['thr_num'], 10))['split_gain'].sum().idxmax()  # highest-gain band
    rows.append({'feature': f, 'n_splits': len(sn),
                 'total_gain': round(w.sum(), 1),
                 'median_thr': round(sn['thr_num'].median(), 4),
                 'gain_wtd_thr': round(gain_wtd_thr, 4),
                 'top_gain_band': str(band)})
display(pd.DataFrame(rows))

# top single split points overall (which feature@threshold buys the most gain)
print("\nTop 15 individual splits by gain:")
display(splits.dropna(subset=['thr_num'])
              .sort_values('split_gain', ascending=False)
              [['split_feature', 'thr_num', 'split_gain', 'count']].head(15).round(4))

# ---- B. shallow surrogate tree -> readable if/then rules for the forest's risk ----
Xm = holdout_df[features + CATS].copy()
for c in CATS: Xm[c] = Xm[c].astype('category')
risk = bad_model.predict_proba(Xm)[:, 1]                 # forest risk score
hot  = (risk > np.quantile(risk, 0.90)).astype(int)      # explain the top-decile-risk region

Xt = holdout_df[features + CATS].copy()
for c in CATS: Xt[c] = Xt[c].astype('category').cat.codes
surr = DecisionTreeClassifier(max_depth=3, min_samples_leaf=500, random_state=0).fit(Xt, hot)
print(f"\nSurrogate tree (depth 3) — explains the forest's top-decile risk | "
      f"fidelity acc={surr.score(Xt, hot):.3f}\n")
print(export_text(surr, feature_names=list(Xt.columns), max_depth=3))

Split thresholds per top driver (gain-weighted, from the actual forest):



,feature,n_splits,total_gain,median_thr,gain_wtd_thr,top_gain_band
0,FlowRatio,929,703836.5,0.7122,0.7162,"(0.675, 0.779]"
1,rtm_sum_1m,235,3453958.6,-1560.8736,-245.9394,"(-12109.197, -1.0000000000000001e-35]"
2,rtda_std_1m,246,1197566.7,47.0202,47.5194,"(-1.869, 186.882]"
3,wind_forecast,799,211632.1,12935.9373,12483.5781,"(8036.136, 10668.14]"
4,rtm_sum_7d,168,501454.5,-499.3133,-370.2992,"(-3159.47, -1.0000000000000001e-35]"
5,rtda_min_3m,160,45341.4,-958.8832,-1585.7796,"(-1955.545, -1.0000000000000001e-35]"
6,rt_nz_cnt_1y,376,106217.8,31.5000,23.9223,"(1.222, 29.3]"
7,rtm_sum_1y,281,75223.9,-10480.9818,-20894.7025,"(-58216.228, -1.0000000000000001e-35]"



Top 15 individual splits by gain:


,split_feature,thr_num,split_gain,count
0,rtm_sum_1m,-122.1639,269978.0,582105
61,rtm_sum_1m,-122.1639,254173.0,582105
122,rtm_sum_1m,-122.1639,239559.0,582105
183,rtm_sum_1m,-122.1639,226014.0,582105
244,rtm_sum_1m,-71.9241,213443.0,582105
305,rtm_sum_1m,-122.1639,201733.0,582105
366,rtm_sum_1m,-71.9241,190865.0,582105
427,rtm_sum_1m,-71.9241,180661.0,582105
488,rt_max_1m,-103.3107,170717.0,582105
549,rtm_sum_1m,-71.9241,162154.0,582105



Surrogate tree (depth 3) — explains the forest's top-decile risk | fidelity acc=0.948

|--- rtm_sum_7d <= -285.29
|   |--- FlowRatio <= 0.79
|   |   |--- short_profit_sum_3d <= -18369.42
|   |   |   |--- class: 1
|   |   |--- short_profit_sum_3d >  -18369.42
|   |   |   |--- class: 0
|   |--- FlowRatio >  0.79
|   |   |--- wind_forecast <= 5706.84
|   |   |   |--- class: 1
|   |   |--- wind_forecast >  5706.84
|   |   |   |--- class: 1
|--- rtm_sum_7d >  -285.29
|   |--- rtm_sum_1m <= -236.92
|   |   |--- FlowRatio <= 0.83
|   |   |   |--- class: 0
|   |   |--- FlowRatio >  0.83
|   |   |   |--- class: 1
|   |--- rtm_sum_1m >  -236.92
|   |   |--- ShadowPrice_sum <= -1119.85
|   |   |   |--- class: 0
|   |   |--- ShadowPrice_sum >  -1119.85
|   |   |   |--- class: 0

